## Example 1: Basic function creation with dynamic signatures

In [ ]:
from makefun import create_function, wraps
import inspect

# Example 1: Basic function creation with dynamic signatures
print("Example 1: Basic Function Creation with Dynamic Signatures")

# Create a function dynamically with a specific signature
# makefun calls func_impl with the arguments BY KEYWORD, so the impl's
# parameter names must match the declared signature's
def multiply_impl(a, b):
    """Multiply two numbers"""
    return a * b

# Create a function with custom signature
multiply = create_function(
    func_signature="multiply(a: int, b: int) -> int",
    func_impl=multiply_impl,
    func_name="multiply"
)

print("Dynamic function created with custom signature")
print(f"Function name: {multiply.__name__}")
print(f"Function signature: {inspect.signature(multiply)}")
print(f"Function docstring: {multiply.__doc__}")

# Test the dynamically created function
result = multiply(5, 3)
print(f"multiply(5, 3) = {result}")
assert result == 15
assert str(inspect.signature(multiply)) == "(a: int, b: int) -> int"

# Verify signature inspection works properly
sig = inspect.signature(multiply)
print(f"\nSignature parameters:")
for name, param in sig.parameters.items():
    print(f"  {name}: {param.annotation.__name__ if param.annotation != inspect.Parameter.empty else 'no annotation'}")
print(f"Return annotation: {sig.return_annotation.__name__ if sig.return_annotation != inspect.Parameter.empty else 'no annotation'}")

# Create function with default parameters
def greet_impl(message, name, greeting="Hello"):
    return f"{greeting}, {name}! {message}"

greet = create_function(
    func_signature="greet(message: str, name: str, greeting: str = 'Hello') -> str",
    func_impl=greet_impl,
    func_name="greet"
)

print(f"\nTesting function with defaults:")
print(f"greet('How are you?', 'Alice'): {greet('How are you?', 'Alice')}")
print(f"greet('Nice to meet you!', 'Bob', 'Hi'): {greet('Nice to meet you!', 'Bob', 'Hi')}")
assert greet('How are you?', 'Alice') == "Hello, Alice! How are you?"
assert greet('Nice to meet you!', 'Bob', 'Hi') == "Hi, Bob! Nice to meet you!"

# Create function with *args and **kwargs
def flexible_impl(*args, **kwargs):
    return f"args: {args}, kwargs: {kwargs}"

flexible = create_function(
    func_signature="flexible(*args, **kwargs) -> str",
    func_impl=flexible_impl,
    func_name="flexible"
)

print(f"\nTesting flexible function:")
print(f"flexible(1, 2, 3, x=10, y=20): {flexible(1, 2, 3, x=10, y=20)}")
assert flexible(1, 2, 3, x=10, y=20) == "args: (1, 2, 3), kwargs: {'x': 10, 'y': 20}"
assert str(inspect.signature(flexible)) == "(*args, **kwargs) -> str"

# Show signature preservation benefits
print(f"\nSignature preservation benefits:")
benefits = [
    "IDE support: Proper autocomplete and type hints in development",
    "Runtime introspection: Functions work correctly with inspect module", 
    "Documentation: Signatures appear correctly in help() and documentation",
    "Type checking: Static type checkers can analyze the signatures",
    "Debugging: Stack traces show meaningful parameter names"
]

for benefit in benefits:
    print(f"  • {benefit}")

# Compare with an equivalent lambda: same parameters, but no name and no annotations
lambda_func = lambda a, b: a * b
print(f"\nComparison with lambda:")
print(f"Dynamic function signature: {inspect.signature(multiply)}")
print(f"Lambda function signature: {inspect.signature(lambda_func)}")

print("Basic function creation completed successfully!")

## Example 2: Advanced decorators with signature preservation

In [ ]:
from makefun import wraps
import functools
import inspect
import time
import logging

# Example 2: Advanced decorators with signature preservation  
print("Example 2: Advanced Decorators with Signature Preservation")

# functools.wraps: inspect.signature() reports the original signature only by following
# __wrapped__; the wrapper itself still takes (*args, **kwargs)
def basic_timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"{func.__name__} took {end - start:.4f} seconds")
        return result
    return wrapper

# makefun.wraps generates a wrapper whose own signature is the original one
def enhanced_timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"{func.__name__} took {end - start:.4f} seconds")
        return result
    return wrapper

# Test functions to decorate
def calculate_sum(numbers: list[int], multiplier: int = 1) -> int:
    """Calculate sum of numbers with optional multiplier"""
    return sum(numbers) * multiplier

def process_data(data: dict, validate: bool = True) -> dict:
    """Process data with optional validation"""
    if validate:
        time.sleep(0.01)  # Simulate validation time
    return {"processed": True, "items": len(data)}

print("Original functions defined")

# Apply decorators
basic_timed = basic_timer(calculate_sum)
enhanced_timed = enhanced_timer(calculate_sum)

print(f"\nSignature comparison:")
print(f"Original: {inspect.signature(calculate_sum)}")
print(f"Basic decorator: {inspect.signature(basic_timed)}")
print(f"Enhanced decorator: {inspect.signature(enhanced_timed)}")
print(f"Basic decorator, not following __wrapped__: {inspect.signature(basic_timed, follow_wrapped=False)}")
print(f"Enhanced decorator, not following __wrapped__: {inspect.signature(enhanced_timed, follow_wrapped=False)}")
expected_sig = "(numbers: list[int], multiplier: int = 1) -> int"
assert str(inspect.signature(basic_timed)) == expected_sig
assert str(inspect.signature(basic_timed, follow_wrapped=False)) == "(*args, **kwargs) -> int"
assert str(inspect.signature(enhanced_timed)) == expected_sig
assert str(inspect.signature(enhanced_timed, follow_wrapped=False)) == expected_sig

# Test functionality
test_numbers = [1, 2, 3, 4, 5]

print(f"\nTesting functionality:")
result1 = basic_timed(test_numbers, multiplier=2)
print(f"Basic result: {result1}")

result2 = enhanced_timed(test_numbers, multiplier=2)  
print(f"Enhanced result: {result2}")
assert result1 == result2 == 30

# The generated signature rejects an unknown keyword before the wrapper body runs
try:
    enhanced_timed(test_numbers, multipler=2)  # misspelled keyword
except TypeError as e:
    print(f"Unknown keyword rejected: {e}")
else:
    raise AssertionError("expected TypeError for an unknown keyword argument")

# Create a logging decorator with signature preservation
def log_calls(logger_name: str = "default"):
    def decorator(func):
        logger = logging.getLogger(logger_name)
        
        @wraps(func)
        def wrapper(*args, **kwargs):
            # Log function call with arguments
            args_repr = [repr(a) for a in args]
            kwargs_repr = [f"{k}={v!r}" for k, v in kwargs.items()]
            signature_str = ", ".join(args_repr + kwargs_repr)
            
            logger.info(f"Calling {func.__name__}({signature_str})")
            
            try:
                result = func(*args, **kwargs)
                logger.info(f"{func.__name__} returned {result!r}")
                return result
            except Exception as e:
                logger.error(f"{func.__name__} raised {type(e).__name__}: {e}")
                raise
        
        return wrapper
    return decorator

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# Apply logging decorator
@log_calls("data_processor")
def analyze_metrics(values: list[float], threshold: float = 0.5) -> dict:
    """Analyze metrics and return statistics"""
    filtered = [v for v in values if v >= threshold]
    return {
        "total": len(values),
        "above_threshold": len(filtered),
        "average": sum(filtered) / len(filtered) if filtered else 0
    }

print(f"\nLogged function signature: {inspect.signature(analyze_metrics)}")

# Test logged function
test_values = [0.1, 0.7, 0.3, 0.9, 0.4, 0.8]
result = analyze_metrics(test_values, threshold=0.4)
print(f"Analysis result: {result}")
assert str(inspect.signature(analyze_metrics)) == "(values: list[float], threshold: float = 0.5) -> dict"
assert result["total"] == 6 and result["above_threshold"] == 4 and abs(result["average"] - 0.7) < 1e-9, result

# Create a caching decorator with signature preservation
def memoize(maxsize: int = 128):
    def decorator(func):
        cache = {}
        cache_info = {"hits": 0, "misses": 0}
        
        @wraps(func)
        def wrapper(*args, **kwargs):
            # Create cache key from arguments
            key = str(args) + str(sorted(kwargs.items()))
            
            if key in cache:
                cache_info["hits"] += 1
                return cache[key]
            
            cache_info["misses"] += 1
            result = func(*args, **kwargs)
            
            # Simple LRU: remove oldest if cache is full
            if len(cache) >= maxsize:
                oldest_key = next(iter(cache))
                del cache[oldest_key]
            
            cache[key] = result
            return result
        
        # Add cache_info method to wrapper
        wrapper.cache_info = lambda: cache_info.copy()
        wrapper.cache_clear = lambda: (cache.clear(), cache_info.update({"hits": 0, "misses": 0}))
        
        return wrapper
    return decorator

# Apply caching decorator
@memoize(maxsize=50)
def fibonacci(n: int) -> int:
    """Calculate fibonacci number (inefficient recursive implementation)"""
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print(f"\nCached function signature: {inspect.signature(fibonacci)}")

# Test caching
print(f"Computing fibonacci(10)...")
result = fibonacci(10)
print(f"Result: {result}")
print(f"Cache info: {fibonacci.cache_info()}")
assert result == 55 and fibonacci.cache_info() == {"hits": 8, "misses": 11}
assert str(inspect.signature(fibonacci)) == "(n: int) -> int"

# Test cache hit
print(f"Computing fibonacci(8) again...")
result = fibonacci(8)  # Should be cached from previous call
print(f"Result: {result}")
print(f"Cache info: {fibonacci.cache_info()}")
assert result == 21 and fibonacci.cache_info() == {"hits": 9, "misses": 11}

# Show decorator capabilities
print(f"\nDecorator capabilities with makefun:")
decorator_features = [
    "Signature preservation: Maintain original function signatures perfectly",
    "Type hint support: Preserve all type annotations for static analysis",
    "IDE integration: Full autocomplete and type checking support",
    "Introspection: Functions work correctly with inspect module",
    "Documentation: Help and docstrings display correctly",
    "Debugging: Stack traces show proper parameter names"
]

for feature in decorator_features:
    print(f"  • {feature}")

print("Advanced decorators completed successfully!")

## Example 3: Function factories and parameterized function generation

In [ ]:
from makefun import create_function, wraps
from typing import Callable, TypeVar, Generic
import inspect
import time

# Example 3: Function factories and parameterized function generation
print("Example 3: Function Factories and Parameterized Function Generation")

# Function factory for creating mathematical operations
def create_math_operation(operation_name: str, operator_func: Callable, operation_symbol: str):
    """Factory to create mathematical operation functions"""
    
    def operation_impl(a, b):
        return operator_func(a, b)
    
    # Create function with proper signature and documentation
    func = create_function(
        func_signature=f"{operation_name}(a: float, b: float) -> float",
        func_impl=operation_impl,
        func_name=operation_name,
        doc=f"Compute {operation_name} of two numbers using {operation_symbol} operator"
    )
    
    return func

# Create various math operations
import operator

operations = {
    "add": (operator.add, "+"),
    "subtract": (operator.sub, "-"),
    "multiply": (operator.mul, "*"),
    "divide": (operator.truediv, "/"),
    "power": (operator.pow, "**"),
    "modulo": (operator.mod, "%")
}

print("Creating mathematical operations dynamically:")

math_functions = {}
for name, (op_func, symbol) in operations.items():
    func = create_math_operation(name, op_func, symbol)
    math_functions[name] = func
    print(f"  Created {name}: {inspect.signature(func)}")

# Test the dynamically created functions
print(f"\nTesting generated functions:")
test_cases = [
    ("add", 10, 5),
    ("subtract", 10, 5),
    ("multiply", 10, 5),
    ("divide", 10, 5),
    ("power", 10, 2),
    ("modulo", 10, 3)
]

expected_results = {"add": 15, "subtract": 5, "multiply": 50, "divide": 2.0, "power": 100, "modulo": 1}
for op_name, a, b in test_cases:
    result = math_functions[op_name](a, b)
    print(f"  {op_name}({a}, {b}) = {result}")
    assert result == expected_results[op_name], (op_name, result)
assert str(inspect.signature(math_functions["add"])) == "(a: float, b: float) -> float"
assert math_functions["power"].__name__ == "power"
assert math_functions["power"].__doc__ == "Compute power of two numbers using ** operator"

# Create validation function factory
def create_validator(field_name: str, validation_func: Callable, error_message: str = None):
    """Factory to create field validation functions"""
    
    def validator_impl(value):
        if not validation_func(value):
            msg = error_message or f"Invalid value for {field_name}"
            raise ValueError(msg)
        return value
    
    func_name = f"validate_{field_name}"
    func = create_function(
        func_signature=f"{func_name}(value) -> object",
        func_impl=validator_impl,
        func_name=func_name,
        doc=f"Validate {field_name} field"
    )
    
    return func

# Create various validators
validators = {
    "email": lambda x: isinstance(x, str) and "@" in x and "." in x,
    "age": lambda x: isinstance(x, int) and 0 <= x <= 150,
    "username": lambda x: isinstance(x, str) and len(x) >= 3 and x.isalnum(),
    "password": lambda x: isinstance(x, str) and len(x) >= 8,
    "phone": lambda x: isinstance(x, str) and len(x.replace("-", "").replace(" ", "")) >= 10
}

print(f"\nCreating validation functions:")

validation_functions = {}
for field, validation_rule in validators.items():
    error_msg = f"Invalid {field} format"
    validator = create_validator(field, validation_rule, error_msg)
    validation_functions[field] = validator
    print(f"  Created {validator.__name__}: {inspect.signature(validator)}")

# Test validators
print(f"\nTesting validators:")

test_data = {
    "email": ["user@example.com", "invalid-email"],
    "age": [25, 200],
    "username": ["alice123", "x"],
    "password": ["securepass123", "weak"],
    "phone": ["555-123-4567", "123"]
}

# The first value of each pair is valid, the second is not
for field, (good_value, bad_value) in test_data.items():
    validator = validation_functions[field]
    assert str(inspect.signature(validator)) == "(value) -> object"
    assert validator(good_value) == good_value
    print(f"  ✓ {field} {good_value!r}: valid")
    try:
        validator(bad_value)
    except ValueError as e:
        assert str(e) == f"Invalid {field} format", e
        print(f"  ✗ {field} {bad_value!r}: {e}")
    else:
        raise AssertionError(f"{field}: {bad_value!r} should be rejected")

# Create API endpoint factory
def create_api_endpoint(endpoint_name: str, method: str, required_params: list, optional_params: dict = None):
    """Factory to create API endpoint handler functions"""
    
    optional_params = optional_params or {}
    
    def endpoint_impl(**kwargs):
        # Validate required parameters
        for param in required_params:
            if param not in kwargs:
                raise ValueError(f"Missing required parameter: {param}")
        
        # Add defaults for optional parameters
        for param, default in optional_params.items():
            if param not in kwargs:
                kwargs[param] = default
        
        # Simulate API processing
        return {
            "endpoint": endpoint_name,
            "method": method,
            "status": "success", 
            "params": kwargs,
            "timestamp": time.time()
        }
    
    # Build signature string
    sig_parts = []
    
    # Required parameters
    for param in required_params:
        sig_parts.append(f"{param}: str")
    
    # Optional parameters with defaults
    for param, default in optional_params.items():
        default_repr = repr(default)
        sig_parts.append(f"{param}: str = {default_repr}")
    
    signature_str = f"{endpoint_name}({', '.join(sig_parts)}) -> dict"
    
    func = create_function(
        func_signature=signature_str,
        func_impl=endpoint_impl,
        func_name=endpoint_name,
        doc=f"{method} endpoint for {endpoint_name}"
    )
    
    return func

# Create API endpoints
print(f"\nCreating API endpoints:")

endpoints = [
    ("get_user", "GET", ["user_id"], {"include_profile": "false"}),
    ("create_post", "POST", ["title", "content"], {"published": "false", "tags": ""}),
    ("update_settings", "PUT", ["user_id"], {"notifications": "true", "theme": "default"})
]

api_functions = {}
for name, method, required, optional in endpoints:
    endpoint_func = create_api_endpoint(name, method, required, optional)
    api_functions[name] = endpoint_func
    print(f"  Created {name}: {inspect.signature(endpoint_func)}")

# Test API endpoints
print(f"\nTesting API endpoints:")

# Test get_user
result1 = api_functions["get_user"](user_id="123", include_profile="true")
print(f"  get_user result: {result1['status']}")

# Test create_post
result2 = api_functions["create_post"](title="Hello World", content="My first post", tags="intro,welcome")
print(f"  create_post result: {result2['status']}")

# Test update_settings with defaults
result3 = api_functions["update_settings"](user_id="456")
print(f"  update_settings result: {result3['status']}")

assert str(inspect.signature(api_functions["create_post"])) == "(title: str, content: str, published: str = 'false', tags: str = '') -> dict"
assert result1["params"] == {"user_id": "123", "include_profile": "true"}, result1
assert result2["params"] == {"title": "Hello World", "content": "My first post", "published": "false", "tags": "intro,welcome"}, result2
assert result3["params"] == {"user_id": "456", "notifications": "true", "theme": "default"}, result3
assert (result3["endpoint"], result3["method"]) == ("update_settings", "PUT")
# A missing required parameter is rejected by the generated signature itself
try:
    api_functions["get_user"]()
except TypeError as e:
    print(f"  get_user() without user_id: {e}")
else:
    raise AssertionError("get_user() should require user_id")

# Show factory pattern benefits
print(f"\nFunction factory benefits:")
factory_benefits = [
    "Code reuse: Generate similar functions with consistent patterns",
    "Type safety: All generated functions have proper type annotations",
    "Documentation: Automatic docstring generation for each function",
    "Maintainability: Single source of truth for function generation logic",
    "Scalability: Easy to generate large numbers of similar functions",
    "Runtime flexibility: Create functions based on configuration or data"
]

for benefit in factory_benefits:
    print(f"  • {benefit}")

print("Function factories completed successfully!")

## Example 4: Signature manipulation and async function support

In [ ]:
from makefun import create_function, wraps, add_signature_parameters, remove_signature_parameters
from inspect import Parameter, signature
import asyncio
import inspect

# Example 4: Signature manipulation and async function support
print("Example 4: Signature Manipulation and Async Function Support")

# Signature modification - adding parameters. makefun's signature helpers take and return
# inspect.Signature objects; wraps(..., new_sig=...) then builds a function with that signature.
def base_processor(data):
    """Process data with basic functionality"""
    return f"Processed: {data}"

enhanced_sig = add_signature_parameters(
    signature(base_processor),
    last=[Parameter("verbose", Parameter.KEYWORD_ONLY, default=False),
          Parameter("format_type", Parameter.KEYWORD_ONLY, default="json")],
)

@wraps(base_processor, new_sig=enhanced_sig)
def enhanced_processor(data, verbose, format_type):
    return base_processor(data) + f" [{format_type}{', verbose' if verbose else ''}]"

print(f"Original signature: {inspect.signature(base_processor)}")
print(f"Enhanced signature: {inspect.signature(enhanced_processor)}")

# Test enhanced function
result = enhanced_processor("test data", verbose=True, format_type="xml")
print(f"Enhanced result: {result}")
assert str(inspect.signature(enhanced_processor)) == "(data, *, verbose=False, format_type='json')"
assert result == "Processed: test data [xml, verbose]"
assert enhanced_processor("x") == "Processed: x [json]"  # the added defaults apply

# Signature modification - removing parameters
def complex_function(a, b, c, d, e=10, f=20):
    """Complex function with many parameters"""
    return a + b + c + d + e + f

simplified_sig = remove_signature_parameters(signature(complex_function), "c", "d", "f")

@wraps(complex_function, new_sig=simplified_sig)
def simplified_function(a, b, e):
    return complex_function(a, b, c=0, d=0, e=e)

print(f"\nOriginal complex signature: {inspect.signature(complex_function)}")
print(f"Simplified signature: {inspect.signature(simplified_function)}")

# Test simplified function
result = simplified_function(1, 2, e=5)
print(f"Simplified result: {result}")
assert str(inspect.signature(simplified_function)) == "(a, b, e=10)"
assert result == 1 + 2 + 0 + 0 + 5 + 20

# Async function creation: a coroutine impl produces a coroutine function
async def async_impl(url, timeout=5.0):
    """Async implementation for web requests"""
    print(f"Fetching {url} with timeout {timeout}s...")
    await asyncio.sleep(0.01)  # Simulate network delay
    return {"url": url, "status": 200, "data": f"Response from {url}", "timeout": timeout}

# Create async function with proper signature
fetch_data = create_function(
    func_signature="fetch_data(url: str, timeout: float = 5.0) -> dict",
    func_impl=async_impl,
    func_name="fetch_data",
    doc="Fetch data from URL asynchronously"
)

print(f"\nAsync function signature: {inspect.signature(fetch_data)}")
print(f"Is coroutine function: {inspect.iscoroutinefunction(fetch_data)}")
assert inspect.iscoroutinefunction(fetch_data)

# Jupyter already runs an event loop, so await at top level (asyncio.run() refuses inside one)
result = await fetch_data("https://api.example.com", timeout=10.0)
print(f"Async result: {result}")
assert result == {"url": "https://api.example.com", "status": 200,
                  "data": "Response from https://api.example.com", "timeout": 10.0}

# Create async decorator that preserves signatures; it retries only the TRANSIENT error types it is
# given, so any other exception (a real bug) propagates on the first attempt
def async_retry(max_attempts: int = 3, delay: float = 1.0, retry_on: tuple = (ConnectionError,)):
    def decorator(func):
        @wraps(func)
        async def wrapper(*args, **kwargs):
            last_exception = None
            
            for attempt in range(max_attempts):
                try:
                    return await func(*args, **kwargs)
                except retry_on as e:
                    last_exception = e
                    print(f"Attempt {attempt + 1} failed: {e}")
                    
                    if attempt < max_attempts - 1:
                        await asyncio.sleep(delay)
            
            raise last_exception
        
        return wrapper
    return decorator

# Apply async decorator to an operation that fails once, then succeeds (deterministic)
attempts = []

@async_retry(max_attempts=2, delay=0.01)
async def unreliable_operation(data: str, fail_first: bool = True) -> str:
    """Operation that fails on its first call"""
    attempts.append(data)
    if fail_first and len(attempts) == 1:
        raise ConnectionError("Transient failure occurred")
    return f"Success: {data}"

print(f"\nAsync decorated function signature: {inspect.signature(unreliable_operation)}")
assert str(inspect.signature(unreliable_operation)) == "(data: str, fail_first: bool = True) -> str"
assert inspect.iscoroutinefunction(unreliable_operation)
retried = await unreliable_operation("payload")
print(f"Retried result: {retried} after {len(attempts)} attempt(s)")
assert retried == "Success: payload" and len(attempts) == 2

# Function signature introspection utilities
def analyze_function_signature(func):
    """Analyze and display detailed function signature information"""
    sig = inspect.signature(func)
    
    analysis = {
        "name": func.__name__,
        "is_async": inspect.iscoroutinefunction(func),
        "parameter_count": len(sig.parameters),
        "has_return_annotation": sig.return_annotation != inspect.Signature.empty,
        "parameters": []
    }
    
    for name, param in sig.parameters.items():
        param_info = {
            "name": name,
            "kind": param.kind.name,
            "has_default": param.default != inspect.Parameter.empty,
            "has_annotation": param.annotation != inspect.Parameter.empty
        }
        
        if param_info["has_default"]:
            param_info["default_value"] = param.default
        
        if param_info["has_annotation"]:
            param_info["annotation"] = param.annotation.__name__ if hasattr(param.annotation, '__name__') else str(param.annotation)
        
        analysis["parameters"].append(param_info)
    
    return analysis

# Analyze various functions
print(f"\nFunction signature analysis:")

functions_to_analyze = [
    enhanced_processor,
    simplified_function,
    fetch_data,
    unreliable_operation
]

for func in functions_to_analyze:
    analysis = analyze_function_signature(func)
    print(f"\nFunction: {analysis['name']}")
    print(f"  Async: {analysis['is_async']}")
    print(f"  Parameters: {analysis['parameter_count']}")
    print(f"  Return annotation: {analysis['has_return_annotation']}")
    
    for param in analysis['parameters']:
        param_str = f"    {param['name']} ({param['kind']})"
        if param['has_annotation']:
            param_str += f": {param['annotation']}"
        if param['has_default']:
            param_str += f" = {param['default_value']}"
        print(param_str)

# Advanced signature manipulation - parameter reordering with makefun.create_function
def create_reordered_function(func, new_parameter_order):
    """Create a function with reordered parameters"""
    original_sig = inspect.signature(func)
    
    # Parameters in the specified order, then any the order did not name
    new_params = [original_sig.parameters[name] for name in new_parameter_order if name in original_sig.parameters]
    new_params += [param for name, param in original_sig.parameters.items() if name not in new_parameter_order]
    new_sig = original_sig.replace(parameters=new_params)
    
    # makefun passes every argument to the impl BY KEYWORD, so it can forward them untouched
    def reordered_impl(**kwargs):
        return func(**kwargs)
    
    return create_function(new_sig, reordered_impl, func_name=f"reordered_{func.__name__}")

# Test parameter reordering (a parameter without a default must stay ahead of the defaulted ones)
def original_func(a, b=2, c=3, d=4):
    return f"a={a}, b={b}, c={c}, d={d}"

reordered_func = create_reordered_function(original_func, ['a', 'd', 'c', 'b'])

print(f"\nParameter reordering:")
print(f"Original: {inspect.signature(original_func)}")
print(f"Reordered: {inspect.signature(reordered_func)}")

# Test both functions: positional arguments now bind in the new order
orig_result = original_func(1, c=30)
reord_result = reordered_func(1, 40, 30)

print(f"Original result: {orig_result}")
print(f"Reordered result: {reord_result}")
assert str(inspect.signature(reordered_func)) == "(a, d=4, c=3, b=2)"
assert reordered_func.__name__ == "reordered_original_func"
assert orig_result == "a=1, b=2, c=30, d=4"
assert reord_result == "a=1, b=2, c=30, d=40"

# Show signature manipulation capabilities
print(f"\nSignature manipulation capabilities:")
manipulation_features = [
    "Parameter addition: Add new parameters to existing functions",
    "Parameter removal: Remove unwanted parameters from functions",
    "Signature preservation: Maintain type hints and defaults during modification",
    "Async support: Full support for async/await function patterns",
    "Parameter reordering: Change parameter order while maintaining functionality",
    "Introspection utilities: Advanced signature analysis and inspection tools"
]

for feature in manipulation_features:
    print(f"  • {feature}")

print("Signature manipulation and async support completed successfully!")

## Example 5: Production patterns and performance optimization

In [ ]:
from makefun import create_function, wraps
import concurrent.futures
import inspect
import threading

# Example 5: Production patterns and performance optimization
print("Example 5: Production Patterns and Performance Optimization")

# Performance-optimized function factory
class OptimizedFunctionFactory:
    """Factory for creating high-performance functions with caching"""
    
    def __init__(self):
        self._function_cache = {}
        self._lock = threading.RLock()
    
    def create_cached_function(self, signature_str: str, implementation, name: str, doc: str = None):
        """Create function with caching to avoid recreation overhead"""
        cache_key = (signature_str, name, id(implementation))
        
        with self._lock:
            if cache_key in self._function_cache:
                return self._function_cache[cache_key]
            
            func = create_function(
                func_signature=signature_str,
                func_impl=implementation,
                func_name=name,
                doc=doc
            )
            
            self._function_cache[cache_key] = func
            return func
    
    def get_cache_stats(self):
        """Get factory cache statistics"""
        with self._lock:
            return {
                "cached_functions": len(self._function_cache),
                "memory_usage": sum(sys.getsizeof(f) for f in self._function_cache.values())
            }

# Create optimized factory
factory = OptimizedFunctionFactory()
print("Optimized function factory created")

# Performance testing: function creation overhead
import sys
import time

def test_implementation(x, y=10):
    return x * y + x

# Test regular creation (recreating each time)
start_time = time.time()
regular_functions = []

for i in range(100):
    func = create_function(
        func_signature=f"test_func_{i}(x: int, y: int = 10) -> int",
        func_impl=test_implementation,
        func_name=f"test_func_{i}"
    )
    regular_functions.append(func)

regular_creation_time = time.time() - start_time

# Test cached creation
start_time = time.time()
cached_functions = []

for i in range(100):
    func = factory.create_cached_function(
        signature_str=f"cached_func_{i}(x: int, y: int = 10) -> int",
        implementation=test_implementation,
        name=f"cached_func_{i}"
    )
    cached_functions.append(func)

cached_creation_time = time.time() - start_time

print(f"\nFunction creation performance:")
print(f"  Regular creation (100 funcs): {regular_creation_time:.4f}s")
print(f"  Cached creation, first request for each of 100 signatures: {cached_creation_time:.4f}s")

# Generated functions behave like the implementation, under their own name and signature
assert regular_functions[5](3) == 33 and regular_functions[5](3, y=2) == 9
assert str(inspect.signature(regular_functions[5])) == "(x: int, y: int = 10) -> int"
assert regular_functions[5].__name__ == "test_func_5"
# A repeated request is answered from the factory cache with the same function object
assert factory.create_cached_function(
    signature_str="cached_func_7(x: int, y: int = 10) -> int",
    implementation=test_implementation,
    name="cached_func_7",
) is cached_functions[7]
assert factory.get_cache_stats()["cached_functions"] == 100

# Memory usage analysis
regular_memory = sum(sys.getsizeof(f) for f in regular_functions)
cached_memory = sum(sys.getsizeof(f) for f in cached_functions)
cache_stats = factory.get_cache_stats()

print(f"\nMemory usage:")
print(f"  Regular functions: {regular_memory:,} bytes")
print(f"  Cached functions: {cached_memory:,} bytes")
print(f"  Factory cache: {cache_stats['cached_functions']} functions")

# Thread-safe function generation
def create_worker_function(worker_id: int, task_type: str):
    """Create worker function for concurrent processing"""
    
    def worker_impl(data, config=None):
        config = config or {}
        
        # Simulate different types of work
        if task_type == "cpu_intensive":
            result = sum(i ** 2 for i in range(data.get("iterations", 1000)))
        elif task_type == "io_intensive":
            time.sleep(data.get("delay", 0.01))
            result = f"IO completed for worker {worker_id}"
        else:
            result = f"Worker {worker_id} processed {data}"
        
        return {
            "worker_id": worker_id,
            "task_type": task_type,
            "result": result,
            "timestamp": time.time()
        }
    
    func = factory.create_cached_function(
        signature_str=f"worker_{worker_id}(data: dict, config: dict = None) -> dict",
        implementation=worker_impl,
        name=f"worker_{worker_id}",
        doc=f"Worker {worker_id} for {task_type} tasks"
    )
    
    return func

# Create worker pool
print(f"\nCreating worker pool:")

workers = []
task_types = ["cpu_intensive", "io_intensive", "general"]

for i in range(12):  # Create 12 workers
    task_type = task_types[i % len(task_types)]
    worker = create_worker_function(i, task_type)
    workers.append(worker)
    print(f"  Created worker {i}: {worker.__name__} ({task_type})")

# Test concurrent execution
def run_concurrent_tasks():
    """Run tasks concurrently using the worker pool"""
    
    tasks = []
    for i, worker in enumerate(workers):
        task_data = {
            "task_id": i,
            "iterations": 500 + (i * 100),
            "delay": 0.005 + (i * 0.001)
        }
        tasks.append((worker, task_data))
    
    # Sequential execution
    start_time = time.time()
    sequential_results = []
    
    for worker, data in tasks:
        result = worker(data)
        sequential_results.append(result)
    
    sequential_time = time.time() - start_time
    
    # Concurrent execution
    start_time = time.time()
    concurrent_results = []
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        futures = [executor.submit(worker, data) for worker, data in tasks]
        concurrent_results = [future.result() for future in futures]
    
    concurrent_time = time.time() - start_time
    
    return {
        "sequential_time": sequential_time,
        "concurrent_time": concurrent_time,
        "speedup": sequential_time / concurrent_time,
        "results_count": len(concurrent_results)
    }

# Run performance test
perf_results = run_concurrent_tasks()
assert perf_results["results_count"] == len(workers) == 12
assert factory.get_cache_stats()["cached_functions"] == 112  # 100 test functions + 12 workers
assert workers[0]({"iterations": 100})["result"] == sum(i ** 2 for i in range(100))
assert workers[2]({"x": 1})["result"] == "Worker 2 processed {'x': 1}"
assert str(inspect.signature(workers[0])) == "(data: dict, config: dict = None) -> dict"

print(f"\nConcurrent execution performance:")
print(f"  Sequential time: {perf_results['sequential_time']:.4f}s")
print(f"  Concurrent time: {perf_results['concurrent_time']:.4f}s")
print(f"  Speedup: {perf_results['speedup']:.2f}x")
print(f"  Tasks completed: {perf_results['results_count']}")

# Production deployment patterns
class FunctionRegistry:
    """Registry for managing dynamically created functions in production"""
    
    def __init__(self):
        self._registry = {}
        self._metrics = {}
        self._lock = threading.RLock()
    
    def register_function(self, func, category: str = "general"):
        """Register a function with metadata"""
        with self._lock:
            func_id = f"{category}:{func.__name__}"
            self._registry[func_id] = {
                "function": func,
                "category": category,
                "signature": str(inspect.signature(func)),
                "registered_at": time.time(),
                "call_count": 0,
                "total_time": 0.0
            }
            
            self._metrics[func_id] = {
                "calls": 0,
                "errors": 0,
                "avg_time": 0.0
            }
            
            return func_id
    
    def get_function(self, func_id: str):
        """Get registered function with usage tracking"""
        with self._lock:
            if func_id not in self._registry:
                raise ValueError(f"Function {func_id} not registered")
            
            return self._create_tracked_wrapper(func_id)
    
    def _create_tracked_wrapper(self, func_id: str):
        """Create wrapper that tracks function usage"""
        func_info = self._registry[func_id]
        original_func = func_info["function"]
        
        @wraps(original_func)
        def tracked_wrapper(*args, **kwargs):
            start_time = time.time()
            
            try:
                result = original_func(*args, **kwargs)
                
                # Update metrics
                execution_time = time.time() - start_time
                with self._lock:
                    metrics = self._metrics[func_id]
                    metrics["calls"] += 1
                    metrics["avg_time"] = (
                        (metrics["avg_time"] * (metrics["calls"] - 1) + execution_time) 
                        / metrics["calls"]
                    )
                
                return result
                
            except Exception as e:
                with self._lock:
                    self._metrics[func_id]["errors"] += 1
                raise
        
        return tracked_wrapper
    
    def get_registry_stats(self):
        """Get comprehensive registry statistics"""
        with self._lock:
            return {
                "total_functions": len(self._registry),
                "categories": list(set(info["category"] for info in self._registry.values())),
                "metrics": self._metrics.copy()
            }

# Test production registry
registry = FunctionRegistry()

print(f"\nProduction function registry:")

# Register some workers
for i, worker in enumerate(workers[:5]):  # Register first 5 workers
    category = "cpu" if i % 2 == 0 else "io"
    func_id = registry.register_function(worker, category)
    print(f"  Registered: {func_id}")

# Test tracked execution
tracked_worker = registry.get_function("cpu:worker_0")

# Run multiple times to see metrics
for i in range(3):
    result = tracked_worker({"iterations": 100})

stats = registry.get_registry_stats()
print(f"\nRegistry statistics:")
print(f"  Total functions: {stats['total_functions']}")
print(f"  Categories: {stats['categories']}")

worker_0_metrics = stats['metrics'].get('cpu:worker_0', {})
print(f"  worker_0 calls: {worker_0_metrics.get('calls', 0)}")
print(f"  worker_0 avg time: {worker_0_metrics.get('avg_time', 0):.4f}s")
assert stats["total_functions"] == 5 and sorted(stats["categories"]) == ["cpu", "io"], stats
assert worker_0_metrics["calls"] == 3 and worker_0_metrics["errors"] == 0, worker_0_metrics
assert str(inspect.signature(tracked_worker)) == str(inspect.signature(workers[0]))

# Show production best practices
print(f"\nProduction best practices:")
best_practices = [
    "Function caching: Cache created functions to avoid recreation overhead",
    "Thread safety: Use locks for concurrent function creation and access",
    "Memory management: Monitor memory usage of dynamically created functions",
    "Performance tracking: Implement usage metrics and performance monitoring",
    "Registry pattern: Centralized function management and discovery",
    "Error handling: Robust error handling and recovery mechanisms",
    "Resource cleanup: Proper cleanup of unused functions and resources"
]

for practice in best_practices:
    print(f"  • {practice}")

print("Production patterns and optimization completed successfully!")